<a href="https://colab.research.google.com/github/obafemitope50/flyrank-internship-ml/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/obafemitope50/flyrank-internship-ml/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane: Refresh / Content Opportunity Scoring**

I'm choosing this lane because it directly extends the workflow I already practiced in Weeks 1-2:
building a transparent baseline, comparing it against a learned model using Precision@K, and
validating honestly with client-holdout splits. My own client-holdout testing this week showed
the hand-written rule's precision dropped sharply once tested on genuinely unseen clients
(0.900 in-sample -> 0.450 holdout at k=20), while a decision tree held up more consistently
and even beat the rule at k=20 under the same fair test. That gap is real, already measured,
and worth exploring further over the next 7 weeks.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

import pandas as pd, numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = df["is_declining_label"].values if "is_declining_label" in df.columns else \
    df["trend_direction"].str.lower().eq("down").astype(int).values

features = ["content_age_days", "days_since_last_update", "impressions_90d", "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df["client_id"]))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_train, y_train)
tree_p20 = precision_at_k(tree.predict_proba(X_test)[:,1], y_test, 20)

stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

hand_rule_test = df["hand_rule_score"].iloc[test_idx].values if "hand_rule_score" in df.columns else None
hr_p20 = precision_at_k(hand_rule_test, y_test, 20) if hand_rule_test is not None else None

print(f"Client-holdout Precision@20 — hand rule: {hr_p20:.3f} | tree: {tree_p20:.3f}")
print(f"Declining rate in full dataset: {y.mean():.3f}")


Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found. You're ready.
Client-holdout Precision@20 — hand rule: 0.450 | tree: 0.600
Declining rate in full dataset: 0.542


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Decision:** Which pages should a FlyRank content editor review first each week, given limited
review capacity.

**Action:** The editor opens the top-ranked pages from the queue and decides whether to refresh,
expand, or leave each one.

**Cost of a wrong call:** A false positive (flagging a healthy page) wastes a few minutes of
editor time. A false negative (missing a genuinely declining page) is likely costlier — the
traffic loss continues silently and compounds the longer it goes unaddressed. This asymmetry
means recall on real declines may matter as much as precision on the top of the queue.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Reference: declining_rate computed above ≈ 0.542 — over half the dataset needs some form of review,
# which is exactly why prioritization (this lane) matters more than a flat "review everything" policy.
print(f"Pages needing prioritization out of full set: {int(y.sum())} of {len(y)}")

Pages needing prioritization out of full set: 16262 of 30000


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

Three numbers support this lane being worth 7 weeks:

1. **54.2% of the 30,000-page dataset is currently labeled declining** — over half the
   inventory needs some form of review, which is exactly the volume problem prioritization
   (this lane) is meant to solve; a flat "review everything" policy isn't realistic at this scale.
2. **My own client-holdout test (Section 1) showed a real, non-trivial gap between methods**:
   hand rule Precision@20 = 0.450 vs. decision tree Precision@20 = 0.600, on clients neither
   method was trained on. That's a genuine, reproducible signal that a learned approach can
   outperform a fixed rule here.
3. **32 distinct clients** are represented in the starter data, giving enough client diversity
   to make a client-holdout validation design meaningful rather than trivial.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# 3. Quick look at the data (2-3 real numbers)
import pandas as pd

df3 = pd.read_csv("data/raw/content_refresh_anonymized.csv")

declining_rate = df3["trend_direction"].str.lower().eq("down").mean()
print(f"Declining rate across full starter dataset: {declining_rate:.3f} "
      f"({int(declining_rate*len(df3))} of {len(df3)} pages)")

n_clients = df3["client_id"].nunique()
print(f"Clients represented: {n_clients}")

print(f"Client-holdout Precision@20 from my own test — hand rule: 0.450 | tree: 0.600 "
      f"(see Section 1)")


Declining rate across full starter dataset: 0.542 (16262 of 30000 pages)
Clients represented: 32
Client-holdout Precision@20 from my own test — hand rule: 0.450 | tree: 0.600 (see Section 1)


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I can claim:** this work is observed and directional. My client-holdout test showed a
decision tree outperformed a hand-written rule at Precision@20, on this specific 30,000-row
starter slice, using a proxy label (`trend_direction == "down"`) rather than a true future
outcome. This is decision-support: a ranked queue to help an editor prioritize review time,
not a guarantee that any specific page is actually declining or that fixing it will recover
traffic.

**What I cannot claim:** I cannot claim I've proven anything about Google's ranking algorithm.
I cannot claim a refresh causes recovery — that requires a causal experiment, not this data.
I cannot claim this result holds on the full warehouse or on new clients beyond the 32
represented here, since even my "client-holdout" test only held out clients that were already
present in this same 30k-row slice. And as this lane matures, I intend to move away from the
`trend_direction` proxy label toward a genuine future-window label (prior 90 days -> next 30
days), since a same-window label risks encoding the answer rather than the outcome I actually
want to predict.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Self-check: confirm the proxy-label risk is understood and features stay leakage-free
leaky_cols = ["trend_direction", "trend_pct"]
used_features = ["content_age_days", "days_since_last_update", "impressions_90d", "avg_position", "ctr", "word_count"]
overlap = set(leaky_cols) & set(used_features)
print(f"Leakage check — features overlapping with the label source: {overlap if overlap else 'none'}")


Leakage check — features overlapping with the label source: none


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.